# Agentic AI Radar — Stage 1: Agentic Web discourse and knowledge graph

**Research objective.** This notebook examines how autonomous, tool-using and multi-agent AI systems are represented in the assigned AI Media Dataset. Media mentions are evidence about reporting, not proof of deployment, adoption or real-world relationships.

## Research questions
1. How does the volume and relative share of Agentic Web coverage change from September 2024 through August 2026?
2. Which publishers, organizations and named AI technologies are prominent in the selected coverage?
3. What interpretable lexical themes occur in the selected articles?
4. Which entities are reported together, and what evidence-backed, article-level co-mentions can be represented as a provenance-preserving graph?

**Methods at a glance.** Inspect the live KaggleHub dataset schema; parse and clean the source text without discarding its raw form; remove exact duplicate text; select articles with an auditable phrase/tag rule; describe coverage; fit TF–IDF/NMF topics; combine spaCy named-entity recognition with a small explicit technology alias lexicon; create only sentence-level `MENTIONED_WITH` co-occurrence edges; validate artifacts and provide a transparent review sample.

**Scope and limitations.** The professor's original assignment was not included in the project files; the project specification and `claude.md` are therefore the available requirements. Dataset version 2 describes a September 2024–September 2026 collection, while the assignment scope ends in August 2026. This notebook analyzes dates September 2024–August 2026 inclusive. Keyword selection is a reproducible proxy, not ground truth. NLP errors, publisher selection, duplicate syndication, uneven monthly coverage and the dataset's own tags can bias findings. The graph encodes article co-mentions only; no business relations are inferred. Stage 2 and Stage 3 are deliberately out of scope.

**Expected contribution.** A reproducible corpus and article-provenance-aware, conservative lexical exploration of the Agentic Web, with reusable tables and a graph suitable for later research. **AI-tool disclosure:** Copilot SDK assisted with code authoring and debugging; data-dependent statements in this notebook are generated by executed cells. Any human annotations added during review are identified separately from model output.

## 1. Reproducibility, acquisition and source audit

The latest dataset is obtained using the required KaggleHub identifier. The loader enumerates available files and inspects the actual CSV headers before mapping fields. `Unnamed: 0` is treated as an identifier only if it exists and is unique; otherwise a deterministic content-based ID is used. The dataset describes an MIT licence; article-level copyright and redistribution terms should still be checked before sharing full-text exports.

In [ ]:
from __future__ import annotations

import ast
import hashlib
import html
import json
import platform
import re
import sys
import unicodedata
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import scipy
import sklearn
import spacy
from IPython.display import display
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

SEED = 20261011
DATASET_REF = "jannalipenkova/ai-media-dataset"
DATE_START = pd.Timestamp("2024-09-01")
DATE_END_EXCLUSIVE = pd.Timestamp("2026-09-01")
SUBSET_RULE_VERSION = "agentic-web-lexicon-v1"
ROOT = next((parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / "PROJECT_SPEC.md").exists()), Path.cwd())
PROCESSED_DIR = ROOT / "data" / "processed"
TABLE_DIR = ROOT / "outputs" / "tables"
FIGURE_DIR = ROOT / "outputs" / "figures"
GRAPH_DIR = ROOT / "outputs" / "graph"
EVALUATION_DIR = ROOT / "outputs" / "evaluations"
for directory in (PROCESSED_DIR, TABLE_DIR, FIGURE_DIR, GRAPH_DIR, EVALUATION_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print({"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
       "scikit_learn": sklearn.__version__, "scipy": scipy.__version__, "networkx": nx.__version__,
       "spacy": spacy.__version__, "seed": SEED})

In [ ]:
dataset_path = Path(kagglehub.dataset_download(DATASET_REF))
dataset_files = sorted(path for path in dataset_path.rglob("*") if path.is_file())
if not dataset_files:
    raise FileNotFoundError(f"KaggleHub returned no files for {DATASET_REF}: {dataset_path}")
file_inventory = pd.DataFrame([
    {"path": str(path.relative_to(dataset_path)), "bytes": path.stat().st_size,
     "suffix": path.suffix.lower()} for path in dataset_files
])
display(file_inventory)
csv_files = [path for path in dataset_files if path.suffix.lower() == ".csv"]
if len(csv_files) != 1:
    raise ValueError(f"Expected one CSV dataset file after inspection; found {len(csv_files)}: {csv_files}")
source_file = csv_files[0]
header = pd.read_csv(source_file, nrows=0, encoding="utf-8", encoding_errors="replace")
source_columns = list(header.columns)
print("Dataset path:", source_file)
print("Observed schema:", source_columns)
print("Source file bytes:", source_file.stat().st_size)

In [ ]:
def normalized_column_name(name: object) -> str:
    """Normalize a source column label for evidence-based field mapping."""
    return re.sub(r"[^a-z0-9]+", "", str(name).casefold())


def find_column(columns: list[str], candidates: tuple[str, ...], required: bool = False) -> str | None:
    """Find an observed source column by a small set of documented aliases."""
    lookup = {normalized_column_name(column): column for column in columns}
    for candidate in candidates:
        if normalized_column_name(candidate) in lookup:
            return lookup[normalized_column_name(candidate)]
    if required:
        raise ValueError(f"Required field not found. Tried {candidates}; observed columns: {columns}")
    return None


column_map = {
    "title": find_column(source_columns, ("title",)),
    "date": find_column(source_columns, ("date", "published_at", "publication_date"), required=True),
    "content": find_column(source_columns, ("content", "text", "article_text", "body"), required=True),
    "source": find_column(source_columns, ("domain", "source", "publisher")),
    "url": find_column(source_columns, ("url", "link")),
    "tags": find_column(source_columns, ("tags", "tag")),
    "source_id": find_column(source_columns, ("article_id", "id", "unnamed: 0")),
}
print("Observed-to-canonical mapping:", column_map)
raw_df = pd.read_csv(source_file, dtype="string", encoding="utf-8", encoding_errors="replace", low_memory=False)
if raw_df.empty:
    raise ValueError(f"Dataset CSV contains no records: {source_file}")
audit = pd.DataFrame({
    "column": raw_df.columns,
    "dtype": raw_df.dtypes.astype(str).values,
    "missing_count": raw_df.isna().sum().values,
    "missing_percent": (raw_df.isna().mean().values * 100).round(3),
})
print("Raw shape:", raw_df.shape)
display(audit)
print("Raw exact-row duplicate count:", int(raw_df.duplicated().sum()))

## 2. Cleaning and canonical article records

The source `content` field is a serialized list of paragraphs in the observed version. Parsing uses `ast.literal_eval` rather than executing source text. Text is preserved in `text_raw`; `text_clean` applies Unicode NFC normalization, HTML entity/tag cleanup, URL removal and whitespace normalization without lowercasing or removing names. Only exact duplicate cleaned article texts are removed (first source occurrence retained); near-duplicates are not automatically removed because syndication variants can be analytically meaningful. Invalid dates and unusable text are counted and excluded, not silently coerced into observations.

In [ ]:
def parse_text_field(value: object) -> tuple[str, str, bool]:
    """Return preserved readable source text, cleaned text, and parse-fallback status."""
    if pd.isna(value):
        return "", "", False
    raw_value = str(value)
    try:
        parsed = ast.literal_eval(raw_value)
    except (ValueError, SyntaxError):
        parsed = raw_value
        parse_fallback = True
    else:
        parse_fallback = False
    if isinstance(parsed, (list, tuple)):
        raw_text = "\n".join(str(part) for part in parsed if part is not None)
    elif isinstance(parsed, str):
        raw_text = parsed
    else:
        raw_text = str(parsed)
    clean = unicodedata.normalize("NFC", html.unescape(raw_text))
    clean = re.sub(r"(?is)<(script|style).*?>.*?</\1>", " ", clean)
    clean = re.sub(r"(?s)<[^>]+>", " ", clean)
    clean = re.sub(r"https?://\S+|www\.\S+", " ", clean)
    clean = re.sub(r"\s+", " ", clean).strip()
    return raw_text, clean, parse_fallback


def stable_article_id(source_id: object, url: object, title: object, text: str) -> str:
    """Use a unique observed source ID, or derive a deterministic fallback hash."""
    if source_id is not None and not pd.isna(source_id) and str(source_id).strip():
        return str(source_id).strip()
    material = "\x1f".join(str(value or "") for value in (url, title, text))
    return hashlib.sha256(material.encode("utf-8", errors="replace")).hexdigest()


def parse_tags(value: object) -> list[str]:
    """Parse a serialized tag list without evaluating executable code."""
    if pd.isna(value):
        return []
    try:
        parsed = ast.literal_eval(str(value))
    except (ValueError, SyntaxError):
        return [part.strip() for part in re.split(r"[,;|]", str(value)) if part.strip()]
    if isinstance(parsed, (list, tuple, set)):
        return [str(item).strip() for item in parsed if str(item).strip()]
    return [str(parsed).strip()] if str(parsed).strip() else []


records = []
parse_fallback_count = 0
missing_text_count = 0
for row in raw_df.to_dict(orient="records"):
    source_content = row[column_map["content"]]
    text_raw, text_clean, parse_fallback = parse_text_field(source_content)
    parse_fallback_count += int(parse_fallback)
    missing_text_count += int(not text_clean)
    title = "" if column_map["title"] is None or pd.isna(row[column_map["title"]]) else str(row[column_map["title"]]).strip()
    raw_date = row[column_map["date"]]
    parsed_date = pd.to_datetime(raw_date, errors="coerce", utc=True)
    source_id = None if column_map["source_id"] is None else row[column_map["source_id"]]
    tags = [] if column_map["tags"] is None else parse_tags(row[column_map["tags"]])
    records.append({
        "article_id": stable_article_id(source_id, row.get(column_map["url"]) if column_map["url"] else None, title, text_clean),
        "title": title, "date": parsed_date, "source": None if column_map["source"] is None else row[column_map["source"]],
        "url": None if column_map["url"] is None else row[column_map["url"]], "tags": tags,
        "text_raw": text_raw, "text_clean": text_clean,
    })
articles_all = pd.DataFrame.from_records(records)
invalid_date_count = int(articles_all["date"].isna().sum())
outside_window_count = int((articles_all["date"].notna() & ~articles_all["date"].between(DATE_START.tz_localize("UTC"), (DATE_END_EXCLUSIVE - pd.Timedelta(nanoseconds=1)).tz_localize("UTC"))).sum())
articles = articles_all.loc[
    articles_all["date"].notna()
    & articles_all["date"].between(DATE_START.tz_localize("UTC"), (DATE_END_EXCLUSIVE - pd.Timedelta(nanoseconds=1)).tz_localize("UTC"))
    & articles_all["text_clean"].ne("")
].copy()
articles["date"] = articles["date"].dt.tz_convert(None)
articles["_text_hash"] = articles["text_clean"].map(lambda text: hashlib.sha256(text.encode("utf-8")).hexdigest())
duplicate_text_count = int(articles["_text_hash"].duplicated().sum())
duplicate_url_count = int(articles["url"].dropna().duplicated().sum())
articles = articles.drop_duplicates("_text_hash", keep="first").drop(columns="_text_hash").reset_index(drop=True)
if articles["article_id"].duplicated().any():
    raise ValueError("Canonical article_id is not unique after cleaning; inspect source identifier mapping.")
cleaning_counts = pd.DataFrame([
    {"step": "source rows", "records": len(raw_df)},
    {"step": "invalid publication date", "records": invalid_date_count},
    {"step": "outside Sep 2024–Aug 2026 window", "records": outside_window_count},
    {"step": "missing or unusable text", "records": missing_text_count},
    {"step": "content parse fallback", "records": parse_fallback_count},
    {"step": "exact duplicate cleaned text removed", "records": duplicate_text_count},
    {"step": "duplicate URL occurrences (reported; not independently dropped)", "records": duplicate_url_count},
    {"step": "final cleaned articles", "records": len(articles)},
])
display(cleaning_counts)
print("Date coverage:", articles["date"].min(), "through", articles["date"].max())
display(articles[["article_id", "title", "date", "source", "text_raw", "text_clean"]].head(3).assign(
    text_raw=lambda frame: frame["text_raw"].str.slice(0, 260),
    text_clean=lambda frame: frame["text_clean"].str.slice(0, 260),
))

## 3. Agentic Web selection and audit

Selection favors precision and interpretability over exhaustive recall: an article qualifies if (a) its title contains an Agentic Web phrase, (b) its body contains at least three matches to the domain phrase list, or (c) the source's tags contain an agent-related tag and the body contains at least one domain phrase. Bare uses of “agent” (for example, customer-service representatives or software agents) do not qualify on their own. The title/body/tag evidence and rule version are saved for every article. This rule can still admit incidental references and miss articles that use different terminology; a stratified sample is exposed for human review and false-negative inspection.

In [ ]:
AGENTIC_PHRASE = re.compile(
    r"\bagentic\b|\b(?:ai|artificial intelligence|llm|large language model)\s+(?:software\s+)?agents?\b|"
    r"\b(?:autonomous|tool[- ](?:using|enabled)|multi[- ]agent|llm[- ]based)\s+agents?\b|"
    r"\bagent\s+(?:framework|workflow|orchestrat|platform)|"
    r"\b(?:agentic\s+(?:workflow|system|framework|web)|multi[- ]agent\s+(?:system|framework|workflow)|"
    r"agent[- ]based\s+(?:ai|artificial intelligence))",
    flags=re.IGNORECASE,
)
AGENT_TAG = re.compile(r"agent", flags=re.IGNORECASE)


def select_agentic_web(row: pd.Series) -> tuple[bool, str, int, int, bool]:
    """Apply the frozen, auditable lexical/tag rule and return its evidence."""
    title_hits = len(AGENTIC_PHRASE.findall(str(row["title"])))
    body_hits = len(AGENTIC_PHRASE.findall(str(row["text_clean"])))
    agent_tag = any(AGENT_TAG.search(tag) for tag in row["tags"])
    selected = bool(title_hits or body_hits >= 3 or (agent_tag and body_hits >= 1))
    reason = "title_phrase" if title_hits else "body_phrase_frequency" if body_hits >= 3 else "agent_tag_and_body_phrase" if agent_tag and body_hits else "not_selected"
    return selected, reason, title_hits, body_hits, agent_tag


selection = articles.apply(select_agentic_web, axis=1, result_type="expand")
selection.columns = ["is_agentic_web", "selection_reason", "title_phrase_hits", "body_phrase_hits", "agent_related_tag"]
articles = pd.concat([articles, selection], axis=1)
articles["subset_rule_version"] = SUBSET_RULE_VERSION
agentic = articles.loc[articles["is_agentic_web"]].copy().reset_index(drop=True)
if agentic.empty:
    raise ValueError("Agentic Web rule selected no articles; inspect field mapping and corpus before proceeding.")
print(f"Selected {len(agentic):,} / {len(articles):,} articles ({len(agentic) / len(articles):.1%})")
display(agentic[["article_id", "title", "source", "selection_reason", "title_phrase_hits", "body_phrase_hits", "agent_related_tag"]].head(12))
display(articles["selection_reason"].value_counts().rename_axis("selection_reason").to_frame("article_count"))

In [ ]:
rng = np.random.default_rng(SEED)
audit_strata = {
    "selected": agentic,
    "near_miss_1_2_body_phrases": articles.loc[~articles["is_agentic_web"] & articles["body_phrase_hits"].between(1, 2)],
    "no_phrase_or_tag_signal": articles.loc[~articles["is_agentic_web"] & articles["body_phrase_hits"].eq(0)],
}
audit_rows = []
for stratum, frame in audit_strata.items():
    if frame.empty:
        continue
    sample = frame.sample(n=min(12, len(frame)), random_state=SEED)
    for row in sample.itertuples(index=False):
        audit_rows.append({
            "article_id": row.article_id, "audit_stratum": stratum, "is_agentic_web_rule": row.is_agentic_web,
            "title": row.title, "source": row.source, "date": row.date,
            "selection_reason": row.selection_reason, "body_phrase_hits": row.body_phrase_hits,
            "review_text_excerpt": (str(row.title) + " — " + str(row.text_clean)[:500]),
            "manual_label": "", "reviewer_notes": "",
        })
selection_review = pd.DataFrame(audit_rows)
print("Fixed-seed selection audit sample; empty manual_label and reviewer_notes are intentionally unscored until reviewed.")
display(selection_review[["article_id", "audit_stratum", "is_agentic_web_rule", "title", "review_text_excerpt"]].head(18))
print("Audit sample size by stratum:")
display(selection_review["audit_stratum"].value_counts())

**Interpretation of selection output.** The selected set is a rule-defined analysis corpus, not a gold-standard Agentic Web classification. The stratified audit worksheet deliberately leaves labels blank rather than presenting unperformed human judgments as evaluation scores. Selection precision/recall require review and are reported as pending until those labels are supplied. Boundary samples are especially informative for estimating missed cases.

## 4. Exploratory data analysis

Monthly counts and monthly share answer different questions: counts describe the volume of selected reporting, while the share normalizes against all cleaned articles in each month. Publisher counts reflect dataset composition, not the media market as a whole. The dataset is date-filtered before aggregation so the extra September 2026 collection month does not enter the analysis.

In [ ]:
articles["month"] = articles["date"].dt.to_period("M").astype(str)
agentic["month"] = agentic["date"].dt.to_period("M").astype(str)
monthly_total = articles.groupby("month").size().rename("all_articles")
monthly_agentic = agentic.groupby("month").size().rename("agentic_articles")
monthly = pd.concat([monthly_total, monthly_agentic], axis=1).fillna(0).astype(int)
monthly["agentic_share"] = monthly["agentic_articles"] / monthly["all_articles"].replace(0, np.nan)
monthly.to_csv(TABLE_DIR / "monthly_coverage.csv", index_label="month")
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True, constrained_layout=True)
monthly["all_articles"].plot(ax=axes[0], color="#536878", marker=".", label="All cleaned articles")
monthly["agentic_articles"].plot(ax=axes[0], color="#d95f02", marker=".", label="Rule-selected Agentic Web")
axes[0].set(title="Monthly article volume (September 2024–August 2026)", ylabel="Articles")
axes[0].legend()
monthly["agentic_share"].mul(100).plot(ax=axes[1], color="#1b9e77", marker=".")
axes[1].set(title="Agentic Web share of cleaned articles", ylabel="Share (%)", xlabel="Publication month")
axes[1].grid(alpha=.25)
fig.savefig(FIGURE_DIR / "monthly_coverage.png", dpi=160)
plt.show()
display(monthly.describe().round(3))
print("Interpretation: use the lower-panel share to distinguish topic coverage from changes in overall corpus volume; the upper panel shows raw media-document volume.")

In [ ]:
source_counts = agentic["source"].fillna("[missing]").value_counts().head(15).sort_values()
fig, ax = plt.subplots(figsize=(10, 7))
source_counts.plot.barh(ax=ax, color="#7570b3")
ax.set(title="Top publisher/domain labels in selected coverage", xlabel="Articles", ylabel="Dataset source/domain")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "top_sources.png", dpi=160)
plt.show()
lengths = agentic["text_clean"].str.split().str.len()
print("Selected article word-count quantiles:")
display(lengths.quantile([0, .25, .5, .75, .9, .99, 1]).rename("words").to_frame())
print("Interpretation: publisher/domain concentration and text-length dispersion describe the corpus sample and should not be interpreted as representative shares of all news media.")

In [ ]:
eda_vectorizer = CountVectorizer(
    lowercase=True, strip_accents="unicode", stop_words="english", ngram_range=(1, 2),
    min_df=5, max_df=.85, max_features=30000, token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z0-9_-]+\b",
)
eda_matrix = eda_vectorizer.fit_transform(agentic["title"].fillna("") + " " + agentic["text_clean"])
term_totals = np.asarray(eda_matrix.sum(axis=0)).ravel()
top_terms = pd.DataFrame({"term": eda_vectorizer.get_feature_names_out(), "count": term_totals}).nlargest(25, "count")
display(top_terms)
top_terms.sort_values("count").plot.barh(x="term", y="count", legend=False, figsize=(9, 8), color="#e6ab02")
plt.title("Frequent unigrams and bigrams in selected articles")
plt.xlabel("Document-term occurrences")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "frequent_terms.png", dpi=160)
plt.show()
print("Interpretation: this is a stopword-filtered frequency baseline; common words and phrases are descriptive signals, not topics or entity counts.")

## 5. Topic modeling (TF–IDF + NMF)

NMF on TF–IDF is chosen as an interpretable, CPU-feasible baseline for a corpus of this size; it avoids a multi-gigabyte embedding model download. Text is lowercased and English stopwords are excluded for this bag-of-words model only; the preserved corpus and NER inputs remain case-preserving. Topic count is selected by inspecting top-word coherence and representative documents across a small predeclared grid (4, 6, 8). The coherence proxy below is average pairwise PMI (normalized PMI) of top terms in document-level presence data; it is a diagnostic, not proof that a topic is meaningful. NMF initialization and seed are fixed.

In [ ]:
topic_vectorizer = TfidfVectorizer(
    lowercase=True, strip_accents="unicode", stop_words="english", ngram_range=(1, 2),
    min_df=5, max_df=.85, max_features=40000, sublinear_tf=True,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z0-9_-]+\b",
)
topic_matrix = topic_vectorizer.fit_transform(agentic["title"].fillna("") + " " + agentic["text_clean"])
feature_names = topic_vectorizer.get_feature_names_out()
binary_matrix = topic_matrix.copy()
binary_matrix.data[:] = 1
document_frequency = np.asarray(binary_matrix.sum(axis=0)).ravel()
n_docs = binary_matrix.shape[0]
term_prob = document_frequency / n_docs


def mean_npmi(term_indices: np.ndarray) -> float:
    """Compute mean NPMI over distinct topic-word pairs present in documents."""
    scores = []
    for position, left in enumerate(term_indices):
        for right in term_indices[position + 1:]:
            co_document_count = float((binary_matrix[:, left].T @ binary_matrix[:, right]).toarray()[0, 0])
            p_xy = co_document_count / n_docs
            if p_xy <= 0:
                scores.append(-1.0)
                continue
            pmi = np.log(p_xy / (term_prob[left] * term_prob[right]))
            scores.append(float(pmi / -np.log(p_xy)))
    return float(np.mean(scores)) if scores else float("nan")


topic_candidates = {}
topic_diagnostics = []
for n_topics in (4, 6, 8):
    model = NMF(n_components=n_topics, init="nndsvda", random_state=SEED, max_iter=400, tol=1e-4)
    weights = model.fit_transform(topic_matrix)
    topic_word_indices = [np.argsort(component)[::-1][:12] for component in model.components_]
    coherence = float(np.mean([mean_npmi(indices) for indices in topic_word_indices]))
    topic_candidates[n_topics] = (model, weights, topic_word_indices, coherence)
    topic_diagnostics.append({"n_topics": n_topics, "reconstruction_error": float(model.reconstruction_err_), "mean_top12_npmi": coherence})
topic_diagnostics = pd.DataFrame(topic_diagnostics)
display(topic_diagnostics)
print("Topic-count selection uses both the NPMI diagnostic and the inspectable topic/document output below; no single score is treated as a gold measure of coherence.")

In [ ]:
N_TOPICS = 6
nmf_model, topic_weights, topic_word_indices, selected_npmi = topic_candidates[N_TOPICS]
topic_assignment = topic_weights.argmax(axis=1)
topic_strength = topic_weights.max(axis=1)
topic_rows = []
for topic_id, indices in enumerate(topic_word_indices):
    top_words = feature_names[indices].tolist()
    members = np.flatnonzero(topic_assignment == topic_id)
    exemplars = sorted(members, key=lambda idx: topic_weights[idx, topic_id], reverse=True)[:3]
    for rank, idx in enumerate(indices, start=1):
        topic_rows.append({"topic_id": topic_id, "rank": rank, "term": feature_names[idx], "term_weight": float(nmf_model.components_[topic_id, idx])})
    print(f"Topic {topic_id} (n={len(members):,}; NPMI={mean_npmi(indices):.3f}): {', '.join(top_words)}")
    for idx in exemplars:
        print("  Representative:", agentic.iloc[idx]["article_id"], "|", agentic.iloc[idx]["title"][:180])
agentic["topic_id"] = topic_assignment
agentic["topic_weight"] = topic_strength
topic_prevalence = agentic.groupby("topic_id").agg(article_count=("article_id", "size"), mean_topic_weight=("topic_weight", "mean")).reset_index()
topic_prevalence["share"] = topic_prevalence["article_count"] / len(agentic)
display(topic_prevalence)
pd.DataFrame(topic_rows).to_csv(TABLE_DIR / "topic_terms.csv", index=False)
topic_prevalence.to_csv(TABLE_DIR / "topic_prevalence.csv", index=False)
agentic[["article_id", "topic_id", "topic_weight"]].to_csv(TABLE_DIR / "topic_assignments.csv", index=False)
print("Interpretation: topic names are deliberately not assigned automatically; inspect representative terms and articles, merge or describe overlapping themes cautiously.")

## 6. Named-entity recognition and conservative normalization

The lightweight English spaCy model supplies ORG, PERSON and PRODUCT candidate spans. A reviewed, transparent alias lexicon adds selected Agentic Web frameworks, protocols, products and models; it does not label every capitalized token as an entity. Aliases are normalized conservatively to stable canonical IDs. NER is run on the title and first 16,000 characters of each selected article to bound CPU/RAM; this truncation can miss later mentions and is recorded in metadata. Entity spans retain offsets into the combined title/newline/text-clean input.

In [ ]:
DOMAIN_ALIASES = {
    "OpenAI": ("OpenAI", "ORGANIZATION", ["OpenAI"]),
    "Anthropic": ("Anthropic", "ORGANIZATION", ["Anthropic"]),
    "Google": ("Google", "ORGANIZATION", ["Google", "Google DeepMind", "DeepMind"]),
    "Microsoft": ("Microsoft", "ORGANIZATION", ["Microsoft"]),
    "Meta": ("Meta", "ORGANIZATION", ["Meta", "Meta AI"]),
    "NVIDIA": ("NVIDIA", "ORGANIZATION", ["NVIDIA", "Nvidia"]),
    "Amazon": ("Amazon", "ORGANIZATION", ["Amazon", "Amazon Web Services", "AWS"]),
    "Salesforce": ("Salesforce", "ORGANIZATION", ["Salesforce"]),
    "Apple": ("Apple", "ORGANIZATION", ["Apple"]),
    "IBM": ("IBM", "ORGANIZATION", ["IBM"]),
    "Oracle": ("Oracle", "ORGANIZATION", ["Oracle"]),
    "Hugging Face": ("Hugging Face", "ORGANIZATION", ["Hugging Face"]),
    "Mistral AI": ("Mistral AI", "ORGANIZATION", ["Mistral AI", "Mistral"]),
    "LangChain": ("LangChain", "FRAMEWORK", ["LangChain"]),
    "LangGraph": ("LangGraph", "FRAMEWORK", ["LangGraph"]),
    "LlamaIndex": ("LlamaIndex", "FRAMEWORK", ["LlamaIndex"]),
    "CrewAI": ("CrewAI", "FRAMEWORK", ["CrewAI", "Crew AI"]),
    "Microsoft AutoGen": ("Microsoft AutoGen", "FRAMEWORK", ["AutoGen", "Autogen"]),
    "Semantic Kernel": ("Semantic Kernel", "FRAMEWORK", ["Semantic Kernel"]),
    "Model Context Protocol": ("Model Context Protocol", "PROTOCOL", ["Model Context Protocol", "MCP"]),
    "Agent2Agent Protocol": ("Agent2Agent Protocol", "PROTOCOL", ["Agent2Agent", "A2A protocol"]),
    "ChatGPT": ("ChatGPT", "AI_PRODUCT", ["ChatGPT"]),
    "Claude": ("Claude", "AI_MODEL", ["Claude"]),
    "Gemini": ("Gemini", "AI_MODEL", ["Gemini"]),
    "GPT-4": ("GPT-4", "AI_MODEL", ["GPT-4", "GPT 4"]),
    "Llama": ("Llama", "AI_MODEL", ["Llama 3", "Llama 4"]),
    "Copilot": ("Copilot", "AI_PRODUCT", ["Copilot", "GitHub Copilot"]),
    "Agentforce": ("Agentforce", "AI_PRODUCT", ["Agentforce", "AgentForce"]),
    "Devin": ("Devin", "AI_PRODUCT", ["Devin"]),
    "Manus": ("Manus", "AI_PRODUCT", ["Manus AI"]),
    "AutoGPT": ("AutoGPT", "FRAMEWORK", ["AutoGPT"]),
    "Smolagents": ("Smolagents", "FRAMEWORK", ["smolagents"]),
    "OpenClaw": ("OpenClaw", "FRAMEWORK", ["OpenClaw", "Clawdbot"]),
    "AI agent": ("AI agent", "AI_TECHNOLOGY", ["AI agent", "AI agents"]),
    "agentic AI": ("agentic AI", "AI_TECHNOLOGY", ["agentic AI"]),
    "multi-agent system": ("multi-agent system", "AI_TECHNOLOGY", ["multi-agent system", "multiagent system"]),
    "tool use": ("tool use", "AI_TECHNOLOGY", ["tool use", "tool-using agent", "tool-using agents"]),
}
ALIAS_LOOKUP = {}
for canonical, (label, entity_type, aliases) in DOMAIN_ALIASES.items():
    for alias in aliases:
        ALIAS_LOOKUP[alias.casefold()] = (label, entity_type)
ALIAS_PATTERNS = [
    (re.compile(r"(?<![\w])" + re.escape(alias) + r"(?![\w])", re.IGNORECASE), canonical, entity_type, alias)
    for alias, (canonical, entity_type) in sorted(ALIAS_LOOKUP.items(), key=lambda item: len(item[0]), reverse=True)
]

try:
    nlp = spacy.load("en_core_web_sm", disable=["tagger", "attribute_ruler", "lemmatizer", "parser"])
except OSError as exc:
    raise RuntimeError("Missing spaCy model en_core_web_sm. Install it with `python -m spacy download en_core_web_sm` before running Stage 1.") from exc
if "sentencizer" not in nlp.pipe_names:
    nlp.add_pipe("sentencizer", before="ner")
print("NER model:", nlp.meta.get("name"), nlp.meta.get("version"), "pipeline:", nlp.pipe_names)

In [ ]:
entity_records = []
article_entities = defaultdict(list)
article_sentences = {}
ner_input_truncated_count = 0
ner_inputs = []
ner_article_ids = []
for row in agentic.itertuples(index=False):
    document = f"{row.title}\n{row.text_clean}"
    if len(document) > 16000:
        ner_input_truncated_count += 1
    ner_inputs.append(document[:16000])
    ner_article_ids.append(row.article_id)


def entity_id(canonical_name: str, entity_type: str) -> str:
    """Create a stable, readable type-prefixed entity identifier."""
    slug = re.sub(r"[^a-z0-9]+", "-", canonical_name.casefold()).strip("-")
    return f"{entity_type.casefold()}:{slug}"


for article_id, text, doc in zip(ner_article_ids, ner_inputs, nlp.pipe(ner_inputs, batch_size=32, n_process=1), strict=True):
    spans = {}
    for ent in doc.ents:
        if ent.label_ in {"ORG", "PERSON", "PRODUCT"} and len(ent.text.strip()) > 1:
            entity_type = {"ORG": "ORGANIZATION", "PERSON": "PERSON", "PRODUCT": "PRODUCT"}[ent.label_]
            canonical = ent.text.strip()
            spans[(ent.start_char, ent.end_char)] = (canonical, entity_type, ent.text, "spacy_en_core_web_sm", 0.65)
    for pattern, canonical, entity_type, surface_form in ALIAS_PATTERNS:
        for match in pattern.finditer(text):
            spans[(match.start(), match.end())] = (canonical, entity_type, match.group(0), "domain_alias_lexicon", 0.90)
    mention_spans = []
    for (start, end), (canonical, entity_type, surface, method, confidence) in sorted(spans.items()):
        identifier = entity_id(canonical, entity_type)
        mention = {"article_id": article_id, "entity_id": identifier, "surface_form": surface,
                   "canonical_name": canonical, "entity_type": entity_type, "start_char": start,
                   "end_char": end, "extraction_method": method, "confidence": confidence}
        entity_records.append(mention)
        article_entities[article_id].append(mention)
        mention_spans.append((start, end, mention))
    sentences = []
    for sentence in doc.sents:
        sent_text = sentence.text.strip()
        if not sent_text:
            continue
        sent_start, sent_end = sentence.start_char, sentence.end_char
        sentence_entities = [mention for start, end, mention in mention_spans if start < sent_end and end > sent_start]
        sentences.append({"text": sent_text, "entities": sentence_entities})
    article_sentences[article_id] = sentences
entities = pd.DataFrame.from_records(entity_records, columns=[
    "article_id", "entity_id", "surface_form", "canonical_name", "entity_type", "start_char", "end_char", "extraction_method", "confidence"
])
if not entities.empty and not entities["entity_id"].str.contains(":", regex=False).all():
    raise ValueError("Entity identifier schema validation failed.")
print(f"Extracted {len(entities):,} entity mentions across {entities['entity_id'].nunique():,} normalized IDs and {len(article_entities):,} articles.")
print(f"NER input was truncated to 16,000 characters for {ner_input_truncated_count:,} of {len(agentic):,} selected articles.")
display(entities.groupby(["entity_type", "extraction_method"]).size().rename("mentions").to_frame().sort_values("mentions", ascending=False).head(20))
display(entities[entities["extraction_method"].eq("domain_alias_lexicon")][["surface_form", "canonical_name", "entity_type"]].drop_duplicates().head(25))

## 7. Relationship extraction: evidence-backed co-occurrence only

No relation classifier or labeled relation benchmark is available. To avoid turning co-mention into a false factual predicate, each relation below is explicitly `MENTIONED_WITH`, `relation_kind=cooccurrence`, and is created only when both normalized entity mentions occur in the same sentence. The sentence, article ID and extraction method are retained. These edges describe textual association in this corpus—not partnership, investment, product use or any verified real-world relationship.

In [ ]:
relation_records = []
seen_article_pairs = set()
for article_id, sentences in article_sentences.items():
    for sentence in sentences:
        unique_mentions = {}
        for mention in sentence["entities"]:
            unique_mentions.setdefault(mention["entity_id"], mention)
        ordered_ids = sorted(unique_mentions)
        for left_pos, subject_id in enumerate(ordered_ids):
            for object_id in ordered_ids[left_pos + 1:]:
                key = (article_id, subject_id, object_id)
                if key in seen_article_pairs:
                    continue
                seen_article_pairs.add(key)
                relation_records.append({
                    "article_id": article_id, "subject_id": subject_id, "predicate": "MENTIONED_WITH",
                    "object_id": object_id, "evidence_text": sentence["text"], "relation_kind": "cooccurrence",
                    "extraction_method": "same_sentence_entity_cooccurrence", "confidence": 0.50,
                })
relations = pd.DataFrame.from_records(relation_records, columns=[
    "article_id", "subject_id", "predicate", "object_id", "evidence_text", "relation_kind", "extraction_method", "confidence"
])
if not relations.empty:
    assert relations["relation_kind"].eq("cooccurrence").all()
    assert relations["predicate"].eq("MENTIONED_WITH").all()
    assert not (relations["subject_id"] == relations["object_id"]).any()
print(f"Extracted {len(relations):,} sentence-level co-occurrence records from {relations['article_id'].nunique() if not relations.empty else 0:,} articles.")
display(relations.head(12))
if relations.empty:
    print("No evidence-backed entity pairs were found; the graph will contain entity nodes without edges.")

## 8. Knowledge graph and focused analysis

The directed NetworkX multigraph stores each article-level co-occurrence as an undirected semantic relation represented in both neighbor directions by `MultiGraph` (one edge per article-supported pair). Node attributes include normalized label/type and mention/article counts; edge attributes preserve support article IDs and evidence examples. Degree centrality therefore measures prominence in this extracted media co-mention network only. The visualization is a limited, readable neighborhood around frequently mentioned entities, not the entire graph.

In [ ]:
graph = nx.MultiGraph()
if not entities.empty:
    node_summary = entities.groupby("entity_id").agg(
        label=("canonical_name", "first"), entity_type=("entity_type", "first"),
        mention_count=("entity_id", "size"), source_article_count=("article_id", "nunique"),
    )
    for identifier, row in node_summary.iterrows():
        graph.add_node(identifier, label=row["label"], entity_type=row["entity_type"],
                       mention_count=int(row["mention_count"]), source_article_count=int(row["source_article_count"]))
if not relations.empty:
    for (subject_id, object_id), group in relations.groupby(["subject_id", "object_id"], sort=False):
        graph.add_edge(
            subject_id, object_id, predicate="MENTIONED_WITH", relation_kind="cooccurrence",
            support_count=int(group["article_id"].nunique()), article_ids=json.dumps(sorted(group["article_id"].unique().tolist())),
            evidence_examples=json.dumps(group["evidence_text"].drop_duplicates().head(3).tolist(), ensure_ascii=False),
            extraction_method="same_sentence_entity_cooccurrence", confidence=0.50,
        )
if not set(relations["subject_id"].unique()).union(relations["object_id"].unique()).issubset(graph.nodes) if not relations.empty else False:
    raise ValueError("Graph edge references an entity absent from the node table.")
components = list(nx.connected_components(nx.Graph(graph))) if graph.number_of_nodes() else []
degree_centrality = nx.degree_centrality(nx.Graph(graph)) if graph.number_of_nodes() > 1 else {node: 0.0 for node in graph.nodes}
graph_summary = pd.DataFrame([{
    "nodes": graph.number_of_nodes(), "article_level_edges": graph.number_of_edges(),
    "connected_components_including_isolates": len(components),
    "isolated_nodes": sum(1 for _, degree in graph.degree() if degree == 0),
    "graph_type": "undirected MultiGraph", "edge_semantics": "same-sentence article co-occurrence; not factual relation",
}])
display(graph_summary)
if graph.number_of_nodes():
    centrality_rows = sorted(degree_centrality.items(), key=lambda item: item[1], reverse=True)[:20]
    centrality = pd.DataFrame([{"entity_id": node, "label": graph.nodes[node]["label"],
                                "entity_type": graph.nodes[node]["entity_type"], "degree_centrality": value,
                                "mention_count": graph.nodes[node]["mention_count"]} for node, value in centrality_rows])
    display(centrality)
    core_ids = [node for node, _ in centrality_rows[:12]]
    neighborhood = set(core_ids)
    for node in core_ids:
        neighborhood.update(graph.neighbors(node))
    view_graph = graph.subgraph(neighborhood).copy()
    if view_graph.number_of_edges() > 160:
        keep_edges = sorted(view_graph.edges(keys=True, data=True), key=lambda edge: edge[3].get("support_count", 1), reverse=True)[:160]
        focused = nx.MultiGraph()
        focused.add_nodes_from(view_graph.nodes(data=True))
        focused.add_edges_from(keep_edges)
        view_graph = focused
    plt.figure(figsize=(14, 10))
    positions = nx.spring_layout(nx.Graph(view_graph), seed=SEED, k=0.8)
    labels = {node: graph.nodes[node]["label"] for node in view_graph.nodes}
    nx.draw_networkx_nodes(view_graph, positions, node_size=430, node_color="#80b1d3", alpha=.9)
    nx.draw_networkx_edges(view_graph, positions, alpha=.22, width=.7)
    nx.draw_networkx_labels(view_graph, positions, labels=labels, font_size=7)
    plt.title("Focused entity co-mention network (edge = same-sentence mention)")
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "focused_knowledge_graph.png", dpi=180)
    plt.show()
    print("Interpretation: high degree identifies entities linked to many other extracted entities in these articles; it is not a measure of real-world importance or verified collaboration.")

## 9. Evaluation and validation

This notebook computes reproducible structural and provenance checks (unique IDs, valid offsets, known graph endpoints, same-article evidence, typed co-occurrence edges, valid topic assignments, export round trips). It does **not** claim gold-standard NER precision/recall or relation accuracy without independent annotations. The fixed-seed selection worksheet is exported for review; a reviewer can fill `manual_label` with `relevant`, `not_relevant` or `uncertain`, and notes. Metrics are calculated only if labels are supplied. This keeps unreviewed model/rule outputs distinct from validated labels.

In [ ]:
evaluation_checks = []
def record_check(name: str, passed: bool, detail: str) -> None:
    evaluation_checks.append({"check": name, "status": "Passed" if passed else "Failed", "detail": detail})


record_check("unique cleaned article IDs", articles["article_id"].is_unique, f"{len(articles):,} rows")
record_check("unique selected article IDs", agentic["article_id"].is_unique, f"{len(agentic):,} rows")
record_check("selected rows satisfy frozen rule", bool(agentic["is_agentic_web"].all()), SUBSET_RULE_VERSION)
record_check("entity article provenance", bool(entities["article_id"].isin(set(agentic["article_id"])).all()) if not entities.empty else True,
             f"{entities['article_id'].nunique() if not entities.empty else 0:,} articles represented")
record_check("entity offsets valid for truncated NER input", bool((entities["start_char"] >= 0).all() and (entities["end_char"] > entities["start_char"]).all()) if not entities.empty else True,
             f"{len(entities):,} mention spans")
record_check("relation predicates explicitly co-occurrence", bool(relations["predicate"].eq("MENTIONED_WITH").all() and relations["relation_kind"].eq("cooccurrence").all()) if not relations.empty else True,
             f"{len(relations):,} article-level relation records")
record_check("relation evidence and article provenance present", bool(relations["evidence_text"].notna().all() and relations["article_id"].isin(set(agentic["article_id"])).all()) if not relations.empty else True,
             "Every edge evidence sentence is linked to a selected article")
record_check("topic assignments in configured range", bool(agentic["topic_id"].between(0, N_TOPICS - 1).all()), f"NMF K={N_TOPICS}")
evaluation_table = pd.DataFrame(evaluation_checks)
display(evaluation_table)
if evaluation_table["status"].eq("Failed").any():
    raise AssertionError("One or more Stage 1 structural/provenance checks failed.")

reviewed = selection_review[selection_review["manual_label"].isin(["relevant", "not_relevant", "uncertain"])].copy()
if reviewed.empty:
    selection_metrics = {"status": "Needs review", "n_reviewed": 0,
                         "note": "No manual labels were supplied; precision/recall are not reported."}
else:
    scored = reviewed[reviewed["manual_label"].isin(["relevant", "not_relevant"])]
    predicted = scored["is_agentic_web_rule"].astype(bool)
    gold = scored["manual_label"].eq("relevant")
    tp = int((predicted & gold).sum()); fp = int((predicted & ~gold).sum()); fn = int((~predicted & gold).sum())
    selection_metrics = {"status": "Computed from supplied manual labels", "n_reviewed": len(scored),
                         "precision": tp / (tp + fp) if tp + fp else None,
                         "recall": tp / (tp + fn) if tp + fn else None,
                         "uncertain_excluded": int(reviewed["manual_label"].eq("uncertain").sum())}
print("Selection evaluation:", selection_metrics)

## 10. Reusable artifacts and round-trip checks

Artifacts use stable relative paths. The article tables contain article text, so review dataset and article copyright/redistribution conditions before distributing them. CSVs retain explicit article IDs and evidence. GraphML stores typed nodes and edges, article provenance and JSON-encoded evidence arrays. The metadata JSON records source identity, source file/version evidence, retrieval time, field mapping, filtering/cleaning decisions, random seed and package versions.

In [ ]:
article_export_columns = ["article_id", "title", "date", "source", "url", "tags", "text_raw", "text_clean",
                          "is_agentic_web", "subset_rule_version", "selection_reason", "title_phrase_hits",
                          "body_phrase_hits", "agent_related_tag", "topic_id", "topic_weight"]
articles[article_export_columns[:-2]].to_parquet(PROCESSED_DIR / "articles_clean.parquet", index=False)
agentic[article_export_columns].to_parquet(PROCESSED_DIR / "agentic_subset.parquet", index=False)
entities.to_csv(TABLE_DIR / "entities.csv", index=False)
relations.to_csv(TABLE_DIR / "relations.csv", index=False)
selection_review.to_csv(EVALUATION_DIR / "stage1_review.csv", index=False)
evaluation_table.to_csv(EVALUATION_DIR / "stage1_structural_checks.csv", index=False)
node_table = pd.DataFrame([
    {"entity_id": node, **attrs} for node, attrs in graph.nodes(data=True)
])
edge_table = pd.DataFrame([
    {"source": source, "target": target, "key": key, **attrs}
    for source, target, key, attrs in graph.edges(keys=True, data=True)
])
node_table.to_csv(TABLE_DIR / "graph_nodes.csv", index=False)
edge_table.to_csv(TABLE_DIR / "graph_edges.csv", index=False)
nx.write_graphml(graph, GRAPH_DIR / "agentic_web.graphml")
metadata = {
    "dataset_ref": DATASET_REF, "kaggle_dataset_version": 2,
    "kaggle_version_metadata": "Version 2; 2026-09-16; version notes: 2026-09-16 version (public Kaggle metadata inspected 2026-10-11)",
    "dataset_filename": source_file.name, "dataset_license_reported": "MIT",
    "retrieved_at_utc": datetime.now(timezone.utc).isoformat(), "dataset_path_not_portable": str(source_file),
    "source_columns": source_columns, "column_mapping": column_map,
    "analysis_date_start_inclusive": str(DATE_START.date()),
    "analysis_date_end_inclusive": str((DATE_END_EXCLUSIVE - pd.Timedelta(days=1)).date()),
    "raw_rows": int(len(raw_df)), "clean_rows": int(len(articles)), "agentic_subset_rows": int(len(agentic)),
    "invalid_dates_removed": invalid_date_count, "outside_window_rows_excluded": outside_window_count,
    "missing_text_rows": missing_text_count, "exact_duplicate_clean_text_removed": duplicate_text_count,
    "duplicate_url_occurrences_reported": duplicate_url_count, "text_parse_fallbacks": parse_fallback_count,
    "selection_rule_version": SUBSET_RULE_VERSION,
    "selection_rule": "title phrase OR >=3 body phrase hits OR agent-related dataset tag AND >=1 body phrase hit",
    "selection_evaluation": selection_metrics,
    "topic_model": {"method": "TF-IDF + NMF", "n_topics": N_TOPICS, "seed": SEED,
                    "mean_top12_npmi": selected_npmi, "candidate_diagnostics": topic_diagnostics.to_dict(orient="records")},
    "ner_model": "spaCy en_core_web_sm", "ner_max_characters": 16000,
    "ner_truncated_articles": ner_input_truncated_count,
    "relation_semantics": "same-sentence MENTIONED_WITH co-occurrence only; no factual predicate inferred",
    "graph_type": "undirected NetworkX MultiGraph",
    "seed": SEED, "python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
    "scikit_learn": sklearn.__version__, "scipy": scipy.__version__, "networkx": nx.__version__, "spacy": spacy.__version__,
    "notebook": "notebooks/stage1_analysis.ipynb",
}
metadata_path = PROCESSED_DIR / "stage1_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
print("Wrote artifacts:")
for path in sorted([*PROCESSED_DIR.glob("*"), *TABLE_DIR.glob("*"), *FIGURE_DIR.glob("*"), *GRAPH_DIR.glob("*"), *EVALUATION_DIR.glob("*")]):
    if path.is_file():
        print(f"{path.relative_to(ROOT)}  {path.stat().st_size:,} bytes")

articles_reloaded = pd.read_parquet(PROCESSED_DIR / "articles_clean.parquet")
agentic_reloaded = pd.read_parquet(PROCESSED_DIR / "agentic_subset.parquet")
entities_reloaded = pd.read_csv(TABLE_DIR / "entities.csv")
relations_reloaded = pd.read_csv(TABLE_DIR / "relations.csv")
graph_reloaded = nx.read_graphml(GRAPH_DIR / "agentic_web.graphml")
metadata_reloaded = json.loads(metadata_path.read_text(encoding="utf-8"))
round_trip_checks = {
    "articles_clean_rows_match": len(articles_reloaded) == len(articles),
    "agentic_subset_rows_match": len(agentic_reloaded) == len(agentic),
    "entity_records_match": len(entities_reloaded) == len(entities),
    "relation_records_match": len(relations_reloaded) == len(relations),
    "graph_node_count_match": graph_reloaded.number_of_nodes() == graph.number_of_nodes(),
    "graph_edge_count_match": graph_reloaded.number_of_edges() == graph.number_of_edges(),
    "metadata_dataset_ref_match": metadata_reloaded["dataset_ref"] == DATASET_REF,
}
print("Artifact round-trip checks:", round_trip_checks)
if not all(round_trip_checks.values()):
    raise AssertionError("One or more exported artifact reload checks failed.")

## 11. Results, discussion and compliance audit

The following summary is assembled only from the current run's data. Interpret monthly values as media coverage in this corpus, topics as exploratory lexical clusters, NER output as unvalidated candidates, and graph edges as article co-mentions. No causal or adoption claims are made. The audit flags annotation-dependent work as needing review rather than passing it by implication.

In [ ]:
peak_count_month = monthly["agentic_articles"].idxmax()
peak_share_month = monthly["agentic_share"].idxmax()
top_entities = entities.groupby(["entity_id", "canonical_name", "entity_type"]).agg(
    mentions=("entity_id", "size"), articles=("article_id", "nunique")
).sort_values(["articles", "mentions"], ascending=False).head(10).reset_index() if not entities.empty else pd.DataFrame()
print("RQ1 — observed coverage:", {
    "clean_articles": len(articles), "selected_articles": len(agentic),
    "selected_share": round(len(agentic) / len(articles), 4),
    "peak_month_by_count": peak_count_month, "peak_count": int(monthly.loc[peak_count_month, "agentic_articles"]),
    "peak_month_by_share": peak_share_month, "peak_share": round(float(monthly.loc[peak_share_month, "agentic_share"]), 4),
})
print("RQ2 — most frequently mentioned normalized entities (not real-world importance):")
display(top_entities)
print("RQ3 — NMF topics: see term lists, representative article titles and prevalence above; names require cautious interpretation.")
print("RQ4 — graph: see node/edge/component counts and focused co-mention view above; edge meaning is textual co-occurrence only.")
print("Evaluation status:", selection_metrics)

compliance_rows = [
    ("Research questions, scope and media-evidence limits", "A; introduction", "Passed", "Notebook introduction"),
    ("Real KaggleHub acquisition and actual schema/file inspection", "B; acquisition/audit", "Passed", f"{len(raw_df):,} rows; {len(source_columns)} observed columns; {source_file.name}"),
    ("Missingness, dates, source fields and duplicate audit", "B; cleaning", "Passed", f"{invalid_date_count} invalid dates; {duplicate_text_count} exact duplicate texts removed"),
    ("Preserved raw text and reproducible cleaning", "C; cleaning", "Passed", f"{len(articles):,} final records; cleaning counts displayed"),
    ("Auditable Agentic Web selection and stratified review worksheet", "D; selection", "Passed", f"{len(agentic):,} rule-selected records; {len(selection_review)} review rows exported"),
    ("Measured selection precision/recall against reviewed labels", "D/J; evaluation", "Needs review" if not reviewed.empty else "Needs review", "Metrics are withheld until independent labels are entered"),
    ("Temporal/source/term EDA with saved figures", "E; EDA", "Passed", "Monthly counts/share, top domains, frequent terms"),
    ("Topic model with representative documents and diagnostic", "F; topics", "Passed", f"TF-IDF/NMF K={N_TOPICS}; NPMI diagnostic={selected_npmi:.3f}"),
    ("NER, conservative aliases and provenance", "G; entities", "Passed", f"{len(entities):,} spans; no labeled precision/recall claim"),
    ("Evidence-backed co-occurrence extraction, not factual relations", "H; relations", "Passed", f"{len(relations):,} same-sentence article-level pairs"),
    ("Typed graph, graph properties and focused visualization", "I; graph", "Passed", f"{graph.number_of_nodes():,} nodes; {graph.number_of_edges():,} edges"),
    ("Structural/provenance validation", "J; validation", "Passed" if evaluation_table["status"].eq("Passed").all() else "Failed", f"{len(evaluation_table)} executable checks"),
    ("NER/relation manual precision/recall audit", "G/H/J; evaluation", "Needs review", "No independent labeled benchmark supplied; outputs remain candidates"),
    ("Reusable exports and reload verification", "K; artifacts", "Passed" if all(round_trip_checks.values()) else "Failed", "Parquet, CSV, GraphML and JSON round trips"),
    ("Results, limitations, future Stage 3 handoff", "L; discussion", "Passed", "Conclusions constrained to observed outputs"),
    ("Professor's verbatim assignment cross-check", "Authoritative source", "Needs review", "Original assignment not present among project files"),
    ("AI-tool disclosure", "Academic integrity", "Passed", "Disclosure included in notebook introduction"),
    ("Stage 2/Stage 3/Streamlit excluded", "Scope guardrail", "Passed", "No later-stage notebook or application created"),
]
compliance = pd.DataFrame(compliance_rows, columns=["requirement", "source_or_section", "status", "evidence_or_remaining_issue"])
display(compliance)
compliance.to_csv(EVALUATION_DIR / "stage1_compliance_audit.csv", index=False)
metadata["compliance_audit"] = compliance.to_dict(orient="records")
metadata_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
print("Stage 1 supports Stage 3 through stable article IDs, article text/metadata, topic assignments, normalized entity provenance, evidence-bearing co-occurrence records and an exported GraphML graph. Stage 2 readiness requires confirming the professor's exact model-training requirements.")